Update:
In version 1: i have started with classic classifer with Multi-Head trick. but results in the LB was too bad comparing by other notebooks. So I take it personally. In this version and next we would try as harder as we can to improve the CV.

What is New:
- scale the train data before training.
- no boost preds.
- use modified balance log loss.
- use oof as loss tracker **[CV 0.23]**.

## Imports

In [ ]:
import numpy as np
import pandas as pd

import random

import seaborn as sns; sns.set()
import matplotlib.pyplot as plt 

from tqdm.auto import tqdm

from sklearn.preprocessing import StandardScaler, MinMaxScaler, RobustScaler

from sklearn.model_selection import StratifiedGroupKFold, KFold

from imblearn.over_sampling import RandomOverSampler, SMOTE
from imblearn.under_sampling import RandomUnderSampler

from sklearn.impute import SimpleImputer

from sklearn.linear_model import LogisticRegression
from sklearn.svm import SVC
from sklearn.ensemble import RandomForestClassifier

from sklearn.metrics import roc_curve, auc

from xgboost import XGBClassifier
from catboost import CatBoostClassifier

## Data Exploration

In [ ]:
MAIN_DIR = "/kaggle/input/icr-identify-age-related-conditions/"

train = pd.read_csv(MAIN_DIR + "train.csv")
test = pd.read_csv(MAIN_DIR + "test.csv")
sub = pd.read_csv(MAIN_DIR + "sample_submission.csv")
greeks = pd.read_csv(MAIN_DIR + "greeks.csv")

In [ ]:
train.head()

In [ ]:
greeks.head()

In [ ]:
train.shape, train.Class.value_counts()

target is unbalanced

In [ ]:
train.isna().sum()

there are some missing values in the dataset, let's explore how do we can impute them

## Data Preproccessing

In [ ]:
# define heads
heads = [
    ['AB', 'AF', 'AH', 'AM', 'AR', 'AX', 'AY', 'AZ'],
    ['BC', 'BD ', 'BN', 'BP', 'BQ', 'BR', 'BZ'], 
    ['CB', 'CC', 'CD ', 'CF', 'CH', 'CL', 'CR', 'CS', 'CU', 'CW '], 
    ['DA', 'DE', 'DF', 'DH', 'DI', 'DL', 'DN', 'DU', 'DV', 'DY'], 
    ['EB', 'EE', 'EG', 'EH', 'EJ', 'EL', 'EP', 'EU'], # EJ is categorical
    ['FC', 'FD ', 'FE', 'FI', 'FL', 'FR', 'FS'], 
    ['GB', 'GE', 'GF', 'GH', 'GI', 'GL'], 
]

n_heads = len(heads)

n_heads

In [ ]:
cat_col = 'EJ'

train[cat_col] = train[cat_col].map({"A":0, "B":1})
test[cat_col] = test[cat_col].map({"A":0, "B":1})

In [ ]:
drop_cols = ["Id", "EJ", "Class"]

feat_cols = [col for col in train.columns if col not in drop_cols]

In [ ]:
target_col =  'Class'
labels = train[target_col].values

## Data Visualization

In [ ]:
b_df = train[heads[1]]

b_df.describe()

In most featuers/cases maximum values are far away from the closest quintile, in this case median is preferred over mean in order to describe the features distributions.

In [ ]:
sns.pairplot(b_df, );

In [ ]:
sns.heatmap(b_df.corr(), annot=True);

there are some linear correlation between heads and itself.

In [ ]:
sns.clustermap(b_df.corr(), metric="correlation", figsize=(8, 8));

## Model

The idea behined MultiHeads is simple, we would use No. of heads as features to predict class/target and ensamble all predictions over multi models (linear, tree and ensemble, etc)

### Metrics

In [ ]:
# define loss
def balanced_log_loss(y_true, y_pred):
    # y_true: correct labels 0, 1
    # y_pred: predicted probabilities of class=1
    # calculate the number of observations for each class
    N_0 = np.sum(1 - y_true)
    N_1 = np.sum(y_true)
    # calculate the weights for each class to balance classes
    w_0 = 1 / N_0
    w_1 = 1 / N_1
    # calculate the predicted probabilities for each class
    p_1 = np.clip(y_pred, 1e-15, 1 - 1e-15)
    p_0 = 1 - p_1
    # calculate the summed log loss for each class
    log_loss_0 = -np.sum((1 - y_true) * np.log(p_0))
    log_loss_1 = -np.sum(y_true * np.log(p_1))
    # calculate the weighted summed logarithmic loss
    # (factgor of 2 included to give same result as LL with balanced input)
    balanced_log_loss = 2*(w_0 * log_loss_0 + w_1 * log_loss_1) / (w_0 + w_1)
    # return the average log loss
    return balanced_log_loss/(N_0+N_1)

# https://www.kaggle.com/code/datafan07/icr-simple-eda-baseline
def balance_logloss(y_true, y_pred):
    
    y_pred = np.stack([1-y_pred,y_pred]).T
    y_pred = np.clip(y_pred, 1e-15, 1-1e-15)
    y_pred / np.sum(y_pred, axis=1)[:, None]
    nc = np.bincount(y_true)
    
    logloss = (-1/nc[0]*(np.sum(np.where(y_true==0,1,0) * np.log(y_pred[:,0]))) - 1/nc[1]*(np.sum(np.where(y_true!=0,1,0) * np.log(y_pred[:,1])))) / 2
    
    return logloss

# balanced preds
def boost_preds(yp):
    c_0, c_1 = yp.sum(axis=0)
    # Weighted probabilities based on class imbalance
    prob = yp * np.array([[1/(c_0 if i==0 else c_1) for i in range(yp.shape[1])]])
    yp_ = prob / np.sum(prob, axis=1, keepdims=1)
    
    return yp_

# https://www.kaggle.com/competitions/icr-identify-age-related-conditions/discussion/412507#2291644
def more_boost(oof, c):
    return c*oof / (1 - oof + c*oof)

### Impute

In [ ]:
all_cols = feat_cols + [cat_col]
imputer = SimpleImputer(strategy="median")
imputer.fit(train[all_cols])

train[all_cols] = imputer.transform(train[all_cols])
test[all_cols] = imputer.transform(test[all_cols])

### Normailization

In [ ]:
# default range (0, 1) but the original range is very large, so i descide to scale it larger range
sc = MinMaxScaler(feature_range=(-1, 1)) 
sc.fit(train[all_cols])

train[all_cols] = sc.transform(train[all_cols])
test[all_cols] = sc.transform(test[all_cols])

### Classifiers

In [ ]:
lr = LogisticRegression(C=5.006, 
                        #class_weight="balanced", 
                        random_state=666, 
                        max_iter=600
                       )

svm = SVC(C=5.005, 
          #class_weight="balanced", 
          probability=True
         )

rfc = RandomForestClassifier(n_estimators=600, 
                             max_depth=8, 
                             random_state=666, 
                             class_weight="balanced", 
                             criterion="log_loss", 
                             max_features=0.6
                            )

cat = CatBoostClassifier(iterations=100, 
                         learning_rate=0.05, 
                         random_seed=666, 
                         verbose=0, 
                         #auto_class_weights='Balanced'
                        )

xgb = XGBClassifier(base_score=0.5, 
                    booster='gbtree', 
                    learning_rate=0.300000012, 
                    max_depth=6, 
                    n_estimators=100, 
                    verbosity=None, 
                    objective="binary:logistic",
                   )

xgb1 = XGBClassifier(n_estimators=100, 
                     max_depth=3, 
                     learning_rate=0.2, 
                     subsample=0.9, 
                     colsample_bytree=0.85
                    )

clfs = [svm, xgb, xgb1, xgb1, xgb1]

### Training

In [ ]:
heads_ = heads + [all_cols] * 13 # len(heads_) == 20
n_splits = 11 

loss = []
result = np.zeros(len(test))

count = 0
oof = np.zeros(len(train))
for rs, head in enumerate(tqdm(heads_, total=len(heads_))):
    head = all_cols ## use all feats. as head every iter.
    print("HEAD", rs) #, head)
    train_df = train[head].values
    test_df = test[head].values
    
    gkf = KFold(n_splits=n_splits, shuffle=True, random_state=rs*rs)
    ids = gkf.split(train_df, labels, groups=greeks.iloc[:, 1:-1].sum(1))

    head_result = []
    head_loss = []
    for idx, (train_idx, val_idx) in enumerate(ids): 
        # select fold
        print("--> FOLD:", idx+1, end=" | ") 
        xr, xt = train_df[train_idx], train_df[val_idx] 
        yr, yt = labels[train_idx], labels[val_idx]
        
        # under sampleing
        NUM_POS = np.bincount(yr)[1]
        sampler = RandomUnderSampler(sampling_strategy={0: int(NUM_POS*1.3), 1: NUM_POS}, random_state=rs*idx)
        xr_, yr = sampler.fit_resample(xr, yr)
        
        # train
        for clf in clfs:
            clf.fit(xr_, yr)
            
            yp = clf.predict_proba(xt)
            oof[val_idx] += yp[:, 1] / len(heads_) / len(clfs)
            count += 1
            
            # test
            result += clf.predict_proba(test_df)[:, 1] / len(heads_) / len(clfs)
            
            # tracking loss
            log_loss = balance_logloss(yt, yp[:, 1]) # 
            loss.append(log_loss)
            
            fpr, tpr, thresholds = roc_curve(yt, yp[:, 1])
            auc_score = auc(fpr, tpr)
            
    print("="*12)
        
    #break

Now training logs looks better.

## Evaluate

In [ ]:
overall_cv = balance_logloss(labels, oof)

overall_cv

In [ ]:
np.min(loss), np.mean(loss)

In [ ]:
bst_cv = np.inf
bst_c = 0

for b in range(1, 50):
    oof_ = more_boost(oof, b)
    cv_loss = balanced_log_loss(train.Class.values, oof_)
    if cv_loss < bst_cv:
        print(b, "=>",cv_loss)
        bst_cv = cv_loss
        bst_c = b

Watch out! good people, we overfitting.

In [ ]:
plt.hist(oof, bins=50, label="oof");
plt.hist(oof_, bins=50, label="more oof");
plt.legend()
plt.show()

In [ ]:
test_pred_1 = result/n_splits #more_boost(result/n_splits, bst_c)
test_pred_0 = 1 - test_pred_1

test_pred_1, test_pred_0

## Submission

In [ ]:
preds_cols = ["class_0", "class_1"]
sub[preds_cols] = np.array([test_pred_0, test_pred_1]).T
sub.to_csv("submission.csv", index=False)

sub

This is a baseline, if you have any farther recommandetions, write it in the comment section.

**Upvote** if you like it, your feedback is highly appreciated

In [ ]:
oof1 = oof

In [ ]:
import keras.backend as K

def keras_balance_logloss(y_true, y_pred):
    y_pred = K.stack([1 - y_pred, y_pred], axis=1)
    y_pred = K.clip(y_pred, K.epsilon(), 1 - K.epsilon())
    y_pred /= K.sum(y_pred, axis=1, keepdims=True)
    nc = K.sum(K.cast(K.equal(y_true, 0), dtype=K.floatx()))

    logloss = (-1 / nc * K.sum(K.cast(K.equal(y_true, 0), dtype=K.floatx()) * K.log(y_pred[:, 0])) -
               (1 - 1 / nc) * K.sum(K.cast(K.not_equal(y_true, 0), dtype=K.floatx()) * K.log(y_pred[:, 1]))) / 2

    return logloss

# Example usage:
y_true = np.array([0, 1, 0, 1, 0])
y_pred = np.array([0.3, 0.7, 0.6, 0.4, 0.9])

loss = keras_balance_logloss(K.variable(y_true), K.variable(y_pred))
print(K.eval(loss))


In [ ]:
import tensorflow as tf
from tensorflow import keras
from tensorflow.keras import layers
from tensorflow.keras.models import Sequential


def model(hidden_sizes=[128, 128, 64]):
    model = Sequential()
    model.add(layers.Dense(hidden_sizes[0], activation='relu', input_shape=(len(all_cols),)))
    
    for size in hidden_sizes[1:]:
        model.add(layers.Dense(size, activation='relu'))
    
    model.add(layers.Dense(1, activation='sigmoid'))
    
    # Compile the model
    opt = keras.optimizers.Adam(lr=0.2)
    model.compile(optimizer=opt, loss=keras_balance_logloss)
    
    return model

In [ ]:
model().summary()

In [ ]:
heads_ = heads + [all_cols] * 3 # len(heads_) == 10
n_splits = 5 

loss = []
result = np.zeros(len(test))

count = 0
oof = np.zeros(len(train))
for rs, head in enumerate(tqdm(heads_, total=len(heads_))):
    head = all_cols ## use all feats. as head every iter.
    print("HEAD", rs) #, head)
    train_df = train[head].values
    test_df = test[head].values
    
    gkf = KFold(n_splits=n_splits, shuffle=True, random_state=rs)
    ids = gkf.split(train_df, labels, groups=greeks.iloc[:, 1:-1].sum(1))

    head_result = []
    head_loss = []
    clfs = [model([128, 64, 64]), model([64, 32, 32])]
    for idx, (train_idx, val_idx) in enumerate(ids): 
        # select fold
        print("--> FOLD:", idx+1, end=" | ") 
        xr, xt = train_df[train_idx], train_df[val_idx] 
        yr, yt = labels[train_idx], labels[val_idx]
        
        # under sampleing
        NUM_POS = np.bincount(yr)[1] # {0: int(NUM_POS*1.3), 1: NUM_POS}
        sampler = RandomOverSampler(sampling_strategy="auto", random_state=rs*idx)
        xr_, yr = sampler.fit_resample(xr, yr)
        
        # train
        for clf in clfs:
            #clf.fit(xr_, yr)
            # Train the model
            clf.fit(xr_, yr, epochs=100, batch_size=64, validation_data=(xt, yt), shuffle=True, verbose=0)
            
            yp = clf.predict(xt, verbose=0)
            oof[val_idx] += yp[:, 0] / len(heads_) / len(clfs)
            count += 1
            
            # test
            result += clf.predict(test_df, verbose=0)[:, 0] / len(heads_) / len(clfs)
            
    # tracking loss
    log_loss = balance_logloss(labels, oof) # 
    loss.append(log_loss)
    print("logLoss:", log_loss)

    #fpr, tpr, thresholds = roc_curve(yt, yp[:, 0])
    #auc_score = auc(fpr, tpr)
            
    print("="*12)
        
    #break

In [ ]:
balance_logloss(labels, more_boost(boost_preds(np.array([1-oof, oof1]).T)[:, 1], 1)), balance_logloss(labels, oof)

In [ ]:
balance_logloss(labels, more_boost(oof1, 1)), balance_logloss(labels, oof1), 

In [ ]:
balance_logloss(labels, (boost_preds(np.array([1-oof, oof1]).T)[:, 1] + oof + oof1)/3)

In [ ]:
balance_logloss(labels, boost_preds(np.array([1-oof1, oof]).T)[:, 1]), balance_logloss(labels, oof)

In [ ]:
balance_logloss(labels, boost_preds(np.array([1-oof1, oof1]).T)[:, 1]), balance_logloss(labels, oof1)

In [ ]:
boost_preds(np.array([1-oof, oof]).T)[:, 1]

In [ ]:
oof

In [ ]:
balance_logloss(labels, (.15*oof + .85*oof1))

In [ ]:
(oof1 - oof).mean()

In [ ]:
1-.15